# MARL Notebook (Local)

This is the **local** version of the MARL Crazyflie notebook for training on a local non-cloud machine. It imports shared logic from the module files in this folder rather than embedding it inline. For the original, self-contained Colab notebook, see `MARL_Crazyflie_cloud.ipynb`.

### Using Jax for speedy reinforcement learning

Motivation: After developing our environment and training schedule, we ran into time and compute constraints
- Could not run more than ~40 envs on our computer, which for many timesteps took hours

Solution: Use Jax (via MuJoCo Playground environments) to speed things up

### Background

Jax compiles Python code into optimized machine code (usually via XLA)
- Traces Python functions, builds computation graph, generates machine code for the device (GPU if available, CPU otherwise)
- Jax can then reuse compiled code, which (combined with GPU speed, when available) makes computations very fast
  - Caveats: Have to make some adjustments (e.g. no inner for loops as they cannot be vectorized by Jax)

Since Jax/JIT traces operations and builds a computation graph, we want to keep things relatively simple to avoid long computations
- Created a new Crazyflie XML without all the collision geoms to speed things up
  - Previously, the physics integrator had to consider all collisions even if we never actually collided with anything
  - We weren't using these geoms anyway, as we end training episodes if drone(s) get too close to the ground or one another ("crash" but no collision)

### Resources

[MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground) with example [Cartpole Balance RL Notebook](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/dm_control_suite.ipynb)

[Brax](https://github.com/google/brax) was used for PPO and as an example to build multi-agent PPO


## Setup

This notebook runs entirely on this machine (no Google Drive mounting, no `!pip install` cells, no Colab-only APIs). Most of its logic comes from the module files that live in this same folder:

| Module | What it holds |
|---|---|
| `constants.py` | Physical/training/reward constants and model save paths |
| `crazyflie_env.py` | The `CrazyflieEnv` MJX environment |
| `math_utils.py` | Quaternion / noise helpers used by the environment |
| `debug_utils.py` | Domain-randomization verification helpers |
| `rollout_utils.py` | `rollout_policy(...)` — run + render a policy rollout |
| `training_plots.py` | Live progress-plot helpers for PPO/MAPPO training |
| `ppo_training.py` | Brax PPO config + train-fn builder (single-agent baseline) |
| `mappo_config.py` | `MAPPOConfig` hyperparameters for the custom MAPPO trainer |
| `mappo_models.py` | Actor/critic Flax models + PPO math for MAPPO |
| `vec_env_utils.py` | Vectorized (multi-env) reset/step helpers |
| `mappo_trainer.py` | The custom MAPPO training loop |
| `evaluation.py` | Batched multi-target/seed policy evaluation |
| `checkpoint_utils.py` | Save/load helpers for MAPPO checkpoints |

**Before running this notebook**, create a local virtual environment from the repo root and install dependencies (`pip install` from inside the notebook fails on most setups with an "externally-managed-environment" error):

```bash
cd MARL-CrazyFlie
python3 -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt
pip install -r code/src/gpu_training/playground_requirements.txt
```

> `playground_requirements.txt` pins a CUDA build of `jax` (`jax[cuda12]`), meant for a cloud/CUDA GPU box. On a machine without an NVIDIA GPU (e.g. a mac), prefer the cloud version of the notebook instead (Jax on CPU is quite slow).

Then select this venv as the notebook's kernel and run the cells below in order.


In [ ]:
import os
import sys
import pathlib

# Make sure the module files in this folder (constants.py, crazyflie_env.py,
# rollout_utils.py, ppo_training.py, mappo_trainer.py, ...) are importable,
# regardless of where Jupyter was launched from.
NOTEBOOK_DIR = pathlib.Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import jax
import mujoco

print("JAX version:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("MuJoCo version:", mujoco.__version__)

# Sanity check the MuJoCo installation
mujoco.MjModel.from_xml_string("<mujoco/>")
print("MuJoCo installation OK.")

# Improves steps/sec on some GPUs (Colab T4s etc)
xla_flags = os.environ.get("XLA_FLAGS", "")
if "--xla_gpu_triton_gemm_any=True" not in xla_flags:
    os.environ["XLA_FLAGS"] = xla_flags + " --xla_gpu_triton_gemm_any=True"


## Load XML Scenes

The `assets_mjx/` folder with the Crazyflie MuJoCo scenes already lives alongside this notebook, so there's no Drive-mount or zip-upload step here — `constants.py` resolves the scene paths relative to itself.


In [ ]:
from constants import SCENE_PATH_1_DRONE, SCENE_PATH_2_DRONES

for path in (SCENE_PATH_1_DRONE, SCENE_PATH_2_DRONES):
    assert os.path.exists(path), f"Scene file not found: {path}"

print("SCENE_PATH_1_DRONE :", SCENE_PATH_1_DRONE)
print("SCENE_PATH_2_DRONES:", SCENE_PATH_2_DRONES)


## Crazyflie Environment

`crazyflie_env.py` already imports everything it needs (constants + math helpers), so using it here is just an import.


In [ ]:
from crazyflie_env import CrazyflieEnv


## Verify Domain Randomization and Rollout Example


In [ ]:
import jax

from debug_utils import model_physics_compare

TEST_SEED = 42

rng = jax.random.PRNGKey(TEST_SEED)
env = CrazyflieEnv()
reset_state = env.reset(rng)
mjx_model_2 = env._mjx_model.replace(
    body_mass=reset_state.info["body_mass"],
    body_inertia=reset_state.info["body_inertia"],
    actuator_gainprm=reset_state.info["actuator_gainprm"],
)
model_physics_compare(env._mjx_model, mjx_model_2, TEST_SEED)


In [ ]:
from rollout_utils import rollout_policy

# Crazyflie Rollout Example (upwards hover dummy policy)
env = CrazyflieEnv()

rollout = rollout_policy(
    env=env,
    params=None,
    apply_fn=None,
    episode_length=500,
    seed=42,
    use_dummy_policy=True,
    random_reset=False,
)


# Initial Jax Testing: Single-Agent PPO Using Brax

The following can be run with one drone or multiple (just 2 at the moment). If one drone, it is standard PPO, with 2 drones it is still "single agent" since the actor and critic see all observations and output actions for both drones, instead of drone actors we have a "dispatcher" controlling both.


## PPO Training


In [ ]:
from mujoco_playground import wrapper

from constants import SCENE_PATH_1_DRONE
from ppo_training import default_ppo_params, build_ppo_train_fn
from training_plots import PPOProgressPlotter

ppo_params = default_ppo_params(num_timesteps=60_000_000)
progress = PPOProgressPlotter(num_timesteps=ppo_params["num_timesteps"])
train_fn = build_ppo_train_fn(ppo_params, progress_fn=progress)

# Create environment with one drone
env = CrazyflieEnv(scene_path=SCENE_PATH_1_DRONE, num_drones=1)

# Train PPO agent - make_inference_fn is our policy function, params are
# network parameters, metrics are training stats. wrap_env_fn wraps env for
# training to match Brax format (e.g. adds device and env dimensions to obs).
make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
progress.print_timing()


## PPO Test Rollout


In [ ]:
env = CrazyflieEnv(scene_path=SCENE_PATH_1_DRONE, num_drones=1)

rollout = rollout_policy(
    env,
    params=params,
    apply_fn=make_inference_fn(params, deterministic=True),
    episode_length=3000,
    seed=42,
    print_logs=False,
    random_reset=False,
    use_brax_policy=True,
)


## Save and Load Brax Model

Saved locally under `rl_models/gpu/` (see `constants.PPO_MODEL_SAVE_PATH`) instead of downloading through the browser.


In [ ]:
from brax.io import model

from constants import PPO_MODEL_SAVE_PATH

os.makedirs(os.path.dirname(PPO_MODEL_SAVE_PATH), exist_ok=True)
model.save_params(PPO_MODEL_SAVE_PATH, params)
print(f"Saved PPO params to {PPO_MODEL_SAVE_PATH}")


## Load Brax Model


In [ ]:
from brax.io import model

from constants import PPO_MODEL_SAVE_PATH

params = model.load_params(PPO_MODEL_SAVE_PATH)
inference_fn = make_inference_fn(params)


# Custom Multi-Agent PPO (MAPPO) for Multiple Agents

True multi-agent RL: actors see & act on their own observations (+ relative pos to other drones), critic sees global observation and computes a global reward. The actors share parameters/policy (the same policy examines each drone's observations and outputs an action for each drone).


## MAPPO Training setup (also run this cell if loading a model)


In [ ]:
from constants import ACTION_SIZE_PER_DRONE
from mappo_config import MAPPOConfig
from vec_env_utils import make_env_and_infer

# Can override fields here, e.g. MAPPOConfig(num_updates=200) for a quick test.
mappo_config = MAPPOConfig()

env, PER_ENV_OBS_DIM, PER_AGENT_OBS_DIM, ACTION_DIM = make_env_and_infer(
    lambda: CrazyflieEnv(num_drones=mappo_config.num_drones)
)


## MAPPO Training


In [ ]:
from mappo_trainer import train
from training_plots import MAPPOProgressPlotter

progress = MAPPOProgressPlotter(num_updates=mappo_config.num_updates)

final_state = train(
    env,
    mappo_config,
    PER_ENV_OBS_DIM,
    PER_AGENT_OBS_DIM,
    ACTION_DIM,
    ACTION_SIZE_PER_DRONE,
    progress_fn=progress,
)


## MAPPO Example Rollout


In [ ]:
import jax.numpy as jnp

env = CrazyflieEnv(num_drones=mappo_config.num_drones)

rollout = rollout_policy(
    env,
    params=final_state.policy_state.params,
    apply_fn=final_state.policy_state.apply_fn,
    episode_length=3000,
    seed=42,
    print_logs=False,
    random_reset=False,
    target=jnp.array([0.0, 0.0, 0.5]),
)


# MAPPO Model Utilities (save/run/test)

Model checkpoints for MAPPO are saved/loaded locally under `rl_models/gpu/` (see `constants.MAPPO_CHECKPOINT_DIR`) instead of the Colab download/upload flow. The load cell restores a checkpoint and runs an evaluation.


### Save Model


In [ ]:
from checkpoint_utils import save_checkpoint
from constants import MAPPO_CHECKPOINT_DIR

ckpt_dir = save_checkpoint(final_state, step=mappo_config.num_updates, ckpt_dir=MAPPO_CHECKPOINT_DIR)
print(f"Saved MAPPO checkpoint to {ckpt_dir}")


### Load Model Checkpoint

Run this (after the "MAPPO Training setup" cell above, so `env`/`mappo_config`/the obs-dim variables exist) if you want to load a previously-saved checkpoint instead of training from scratch.


In [ ]:
import jax

from checkpoint_utils import load_checkpoint
from constants import ACTION_SIZE_PER_DRONE, MAPPO_CHECKPOINT_DIR
from mappo_trainer import create_train_state

init_rng = jax.random.PRNGKey(0)
init_state, _, _ = create_train_state(
    init_rng, mappo_config, PER_AGENT_OBS_DIM, PER_ENV_OBS_DIM, ACTION_SIZE_PER_DRONE
)
loaded_state = load_checkpoint(init_state, MAPPO_CHECKPOINT_DIR)

print("MAPPO checkpoint loaded.")


## Test Loaded MAPPO Model for Varying Targets & Seeds


In [ ]:
import jax.numpy as jnp

from evaluation import eval_targets_with_seeds, print_results_table
from mappo_models import make_deterministic_policy

jit_inference = make_deterministic_policy(
    loaded_state.policy_state.apply_fn, loaded_state.policy_state.params
)

# (Assuming we trained with TRAINING_POS_RANGE of 2.0m)
test_targets = [
    # Targets within TRAINING_POS_RANGE
    jnp.array([0.0, 0.0, 0.2]),
    jnp.array([0.3, 0.0, 1.0]),
    jnp.array([-0.3, 0.0, 1.0]),
    jnp.array([0.0, 0.3, 1.0]),
    jnp.array([0.0, -0.3, 1.0]),
    jnp.array([0.5, 0.5, 1.2]),
    jnp.array([-0.5, 0.5, 1.2]),
    jnp.array([0.5, -0.5, 1.2]),
    jnp.array([-0.5, -0.5, 1.2]),
    jnp.array([0.0, 0.0, 2.0]),
    jnp.array([1.0, 0.0, 2.0]),
    jnp.array([-1.0, 0.0, 2.0]),
    jnp.array([0.0, 1.0, 2.0]),
    jnp.array([0.0, -1.0, 2.0]),
    jnp.array([2.0, 2.0, 2.0]),
    jnp.array([-2.0, 2.0, 2.0]),
    jnp.array([2.0, -2.0, 2.0]),
    jnp.array([-2.0, -2.0, 2.0]),
    # Targets outside of TRAINING_POS_RANGE
    jnp.array([0.0, 0.0, 3.0]),
    jnp.array([-3.0, 0.0, 3.0]),
    jnp.array([0.0, -3.0, 3.0]),
    jnp.array([3.0, 3.0, 3.0]),
    jnp.array([5.0, 5.0, 5.0]),
    jnp.array([7.0, 7.0, 7.0]),
]

# Test longer episode length (assuming we trained with 1500)
TEST_EPISODE_LENGTH = 3000
NUM_SEEDS = 64

print(f"Testing each target with {NUM_SEEDS} seeds\n")

returns, steps = eval_targets_with_seeds(
    env,
    jit_inference,
    mappo_config.num_drones,
    PER_AGENT_OBS_DIM,
    test_targets,
    num_seeds=NUM_SEEDS,
    episode_length=TEST_EPISODE_LENGTH,
)

print_results_table(test_targets, returns, steps, TEST_EPISODE_LENGTH)


## Visualize a Model Rollout for a Given Target and Seed


In [ ]:
TEST_TARGET = jnp.array([0.0, 0.0, 3.0])
TEST_SEED = 42

# Inspect domain randomization for the drone physics, given the seed
rng = jax.random.PRNGKey(TEST_SEED)
reset_state = env.reset(rng)
mjx_model_2 = env._mjx_model.replace(
    body_mass=reset_state.info["body_mass"],
    body_inertia=reset_state.info["body_inertia"],
    actuator_gainprm=reset_state.info["actuator_gainprm"],
)
model_physics_compare(env._mjx_model, mjx_model_2, TEST_SEED)

print("\n")

# Run and visualize rollout
rollout = rollout_policy(
    env,
    params=loaded_state.policy_state.params,
    apply_fn=loaded_state.policy_state.apply_fn,
    episode_length=3000,
    seed=TEST_SEED,
    print_logs=False,
    random_reset=False,
    target=TEST_TARGET,
)
